# Pivot Gap Analysis — Reclaimable Corner Acreage from Orbit

**The opportunity.** A center-pivot sprinkler sweeps a circle inside a
square quarter-section. The four corners — roughly **35 acres per field**
on a standard 160-ac quarter-section with a 400 m pivot — sit
unirrigated. Across the U.S. High Plains that's tens of millions of
idle acres. Ag-tech platforms, precision-irrigation OEMs, and
carbon/regen programs all need one thing: **which corners are still
available, and how much acreage is on the table at each?**

This notebook is the RasterFlow pipeline that answers it:

1. **Detect field boundaries** with RasterFlow + Fields of the World on
   Sentinel-2 imagery — one API call, produces the outer square.
2. **Detect the irrigated circle** via NDVI thresholding on the same
   scenes — the green disk is the pivot footprint.
3. **Subtract** — `ST_Difference(field, circle)` = the corner.
4. **Score** — corner acreage, % of field, per-field economics.
5. **Ship** — one Iceberg table + layered GeoJSON for the agronomy
   dashboard.

**AOI.** Haskell County, Kansas — High Plains Ogallala country, roughly
85% irrigated cropland, mostly quarter-section center-pivots.

> **Demo scope.** The RasterFlow API calls in §3-§4 are the production
> pattern (each ~30 min for a county AOI, validated in
> `Analyzing_Data/RasterFlow_FTW.ipynb`). For a runnable demo, §5
> constructs a realistic 12-field quarter-section grid in Haskell with
> a mix of pivot configurations (full, reduced, already-retrofit) so
> the corner math in §6 runs in seconds. Swap in the real FTW +
> NDVI-vectorized layers and the rest of the pipeline is unchanged.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import json

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

TARGET_DATABASE = "pivot_gap_demo"
TARGET_TABLE    = "haskell_corner_inventory"
TARGET_FQN      = f"org_catalog.{TARGET_DATABASE}.{TARGET_TABLE}"

# CONUS Albers Equal Area — use for accurate acreage computation across
# the High Plains. Web Mercator inflates areas by ~60% at 37°N.
AREA_CRS = "EPSG:5070"

## 2. AOI

Define the study area. Real runs use Haskell County's admin polygon
(via `wkls`); here a bbox is enough.

In [ ]:
AOI_BBOX = (-100.98, 37.56, -100.83, 37.73)   # (min_lon, min_lat, max_lon, max_lat)

aoi_df = sedona.sql(f"""
    SELECT ST_MakeEnvelope(
        {AOI_BBOX[0]}, {AOI_BBOX[1]}, {AOI_BBOX[2]}, {AOI_BBOX[3]}, 4326
    ) AS geometry, 'Haskell County, KS (partial)' AS label
""")
aoi_df.createOrReplaceTempView("aoi")

aoi_acres = sedona.sql(f"""
    SELECT ROUND(ST_Area(ST_Transform(geometry, 'EPSG:4326', '{AREA_CRS}')) / 4046.86, 0)
           AS aoi_acres
    FROM aoi
""").first().aoi_acres
print(f"AOI: {aoi_acres:,.0f} acres")

## 3. Stage 1 — RasterFlow Field Boundary Detection (FTW)

The production call is a one-liner that fetches Sentinel-2 L2A, builds
seasonal mosaics, runs the Fields of the World segmentation model, and
vectorizes the `field` class. Runtime ~30 min per county AOI.

```python
from rasterflow_remote import RasterflowClient
from rasterflow_remote.data_models import ModelRecipes, VectorizeMethodEnum
from datetime import datetime

rf = RasterflowClient()

ftw_out = rf.build_and_predict_mosaic_recipe(
    aoi          = aoi_path,            # GeoParquet path on S3
    start        = datetime(2023, 1, 1),
    end          = datetime(2024, 1, 1),
    crs_epsg     = 3857,
    model_recipe = ModelRecipes.FTW,
)
ftw_vectors = rf.vectorize_mosaic(
    store           = ftw_out.first_row_mosaic,
    features        = ['field', 'field_boundaries'],
    threshold       = 0.5,
    vectorize_method= VectorizeMethodEnum.SEMANTIC_SEGMENTATION_RASTERIO,
    vectorize_config= {'stats': True},
)

fields_df = sedona.read.format('geoparquet').load(ftw_vectors.uri) \
    .filter("label = 'field'") \
    .withColumn('geometry', f.expr('ST_MakeValid(geometry)'))
```

This lands a DataFrame of polygon `geometry` (the outer field shape) +
`score_mean` per field.

## 4. Stage 2 — RasterFlow Irrigated-Circle Detection

Center pivots irrigate a crop that's noticeably greener than the
unirrigated corner ground around it. A simple NDVI threshold on the
same Sentinel-2 mosaic isolates the irrigated footprint, which is then
vectorized into circle polygons.

```python
# (a) Load scenes
scenes = sedona.read.format('stac').load(
    'https://earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a'
)

# (b) NDVI per scene — single map-algebra call over Red (B04) + NIR (B08)
ndvi = scenes.selectExpr(
    'id', 'datetime',
    "RS_MapAlgebra(rast, 'FLOAT32', "
    "   'out = (rast[7] - rast[3]) / (rast[7] + rast[3]);') AS ndvi"
)

# (c) Growing-season max NDVI — stable signal for the pivot footprint
peak_ndvi = ndvi.filter("datetime BETWEEN '2023-06-01' AND '2023-09-15'") \
                .selectExpr('RS_Union_Aggr(ndvi, "max") AS peak')

# (d) Threshold + vectorize — fields irrigating through July show NDVI > 0.6
circles_df = rf.vectorize_mosaic(
    store            = peak_ndvi_store,
    features         = ['irrigated'],
    threshold        = 0.60,
    vectorize_method = VectorizeMethodEnum.SEMANTIC_SEGMENTATION_RASTERIO,
)
```

The resulting `circles_df` carries one polygon per active pivot with a
near-circular footprint — the green disk.

## 5. Demo Geometries — 12 Quarter-Section Grid

Construct a realistic 4×3 grid of quarter-section fields (~152 ac each
at this latitude) in the southern half of the AOI, with three pivot
configurations:

- **FULL_STANDARD** (×7) — 400 m radius, the baseline deployment.
- **REDUCED** (×3) — 350 m radius, older / legacy pivot — biggest
  corner upside.
- **EXTENDED** (×2) — 430 m radius, already retrofit with a corner
  sprayer — reference for "fully reclaimed" corners.

In [ ]:
# Quarter-section footprint at ~37.6° N: ~0.009 deg lon × ~0.007 deg lat ≈ 152 ac
LON0, LAT0 = -100.975, 37.570
DLON, DLAT = 0.009, 0.007

BUFFER_BY_TYPE = {
    "FULL_STANDARD": 0.0040,    # ~400 m
    "REDUCED":       0.0036,    # ~350 m
    "EXTENDED":      0.0044,    # ~430 m
}

# 4 columns × 3 rows, with pivot type assigned per cell
layout = [
    # (col, row, pivot_type, owner)
    (0, 0, "FULL_STANDARD", "Acme Farms LLC"),
    (1, 0, "REDUCED",       "Acme Farms LLC"),
    (2, 0, "FULL_STANDARD", "Prairie Holdings"),
    (3, 0, "EXTENDED",      "Prairie Holdings"),
    (0, 1, "FULL_STANDARD", "Sunflower Ag Co"),
    (1, 1, "REDUCED",       "Sunflower Ag Co"),
    (2, 1, "FULL_STANDARD", "Sunflower Ag Co"),
    (3, 1, "FULL_STANDARD", "Acme Farms LLC"),
    (0, 2, "EXTENDED",      "Prairie Holdings"),
    (1, 2, "REDUCED",       "Great Plains LP"),
    (2, 2, "FULL_STANDARD", "Great Plains LP"),
    (3, 2, "FULL_STANDARD", "Great Plains LP"),
]

rows = []
for i, (col, row, pivot_type, owner) in enumerate(layout, start=1):
    min_lon = LON0 + col * DLON
    min_lat = LAT0 + row * DLAT
    max_lon = min_lon + DLON
    max_lat = min_lat + DLAT
    center_lon = (min_lon + max_lon) / 2
    center_lat = (min_lat + max_lat) / 2
    rows.append((
        f"FLD-{i:03d}", pivot_type, owner,
        float(min_lon), float(min_lat), float(max_lon), float(max_lat),
        float(center_lon), float(center_lat),
        float(BUFFER_BY_TYPE[pivot_type]),
    ))

layout_schema = StructType([
    StructField("field_id",      StringType()),
    StructField("pivot_type",    StringType()),
    StructField("owner",         StringType()),
    StructField("min_lon",       DoubleType()),
    StructField("min_lat",       DoubleType()),
    StructField("max_lon",       DoubleType()),
    StructField("max_lat",       DoubleType()),
    StructField("center_lon",    DoubleType()),
    StructField("center_lat",    DoubleType()),
    StructField("pivot_radius_deg", DoubleType()),
])

layout_df = sedona.createDataFrame(rows, layout_schema)
layout_df.createOrReplaceTempView("field_layout")

geoms_df = sedona.sql("""
    SELECT
        field_id, pivot_type, owner,
        ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326) AS field_sq,
        ST_Buffer(ST_Point(center_lon, center_lat), pivot_radius_deg) AS pivot_circle
    FROM field_layout
""").cache()
geoms_df.createOrReplaceTempView("fields_with_pivots")

print(f"Generated {geoms_df.count()} quarter-section fields")
geoms_df.select("field_id", "pivot_type", "owner").show(truncate=False)

## 6. Compute Corners via `ST_Difference`

The whole pivot-gap thesis is one line of SQL: `ST_Difference(field_sq,
pivot_circle)`. Project to CONUS Albers (`EPSG:5070`) for true-acre
areas.

In [ ]:
corners_df = sedona.sql(f"""
    SELECT
        field_id, pivot_type, owner,
        field_sq,
        pivot_circle,
        ST_Difference(field_sq, pivot_circle)  AS corners_geom,
        ROUND(
            ST_Area(ST_Transform(field_sq,     'EPSG:4326','{AREA_CRS}')) / 4046.86, 2
        ) AS field_acres,
        ROUND(
            ST_Area(ST_Transform(pivot_circle, 'EPSG:4326','{AREA_CRS}')) / 4046.86, 2
        ) AS irrigated_acres,
        ROUND(
            ST_Area(ST_Transform(
                ST_Difference(field_sq, pivot_circle),
                'EPSG:4326','{AREA_CRS}'
            )) / 4046.86, 2
        ) AS corner_acres
    FROM fields_with_pivots
""").cache()
corners_df.createOrReplaceTempView("corner_inventory")

corners_df.select("field_id", "pivot_type", "owner",
                  "field_acres", "irrigated_acres", "corner_acres") \
          .orderBy(f.col("corner_acres").desc()) \
          .show(truncate=False)

## 7. Economic Scoring — Is a Retrofit Worth It?

A corner-sprayer retrofit runs ~$15 k per field. If corner ground
produces ~$250/ac/yr of dryland row-crop margin uplift once equipped,
the retrofit pays back in `$15,000 / (corner_acres × $250)` years. Rank
fields by payback period to produce the retrofit shortlist.

In [ ]:
RETROFIT_COST_PER_FIELD = 15000.0
ANNUAL_UPLIFT_PER_ACRE  =   250.0

scored_df = sedona.sql(f"""
    SELECT
        field_id, pivot_type, owner,
        field_acres, irrigated_acres, corner_acres,
        ROUND(corner_acres / field_acres, 3) AS reclaimable_fraction,
        ROUND(corner_acres * {ANNUAL_UPLIFT_PER_ACRE}, 0)
            AS annual_uplift_usd,
        CASE
            WHEN corner_acres > 0 THEN
                ROUND({RETROFIT_COST_PER_FIELD}
                      / (corner_acres * {ANNUAL_UPLIFT_PER_ACRE}), 2)
            ELSE NULL
        END AS payback_years,
        CASE
            WHEN corner_acres >= 45 THEN 'A_PRIORITY'
            WHEN corner_acres >= 30 THEN 'B_PRIORITY'
            WHEN corner_acres >= 20 THEN 'C_REVIEW'
            ELSE                        'D_ALREADY_EFFICIENT'
        END AS retrofit_tier
    FROM corner_inventory
""")
scored_df.createOrReplaceTempView("scored_corners")

scored_df.orderBy(f.col("corner_acres").desc()) \
         .show(truncate=False)

# Portfolio rollup by owner
sedona.sql("""
    SELECT
        owner,
        COUNT(*)                           AS field_count,
        ROUND(SUM(field_acres), 0)         AS total_field_acres,
        ROUND(SUM(corner_acres), 1)        AS reclaimable_acres,
        ROUND(SUM(annual_uplift_usd), 0)   AS annual_uplift_usd
    FROM scored_corners
    GROUP BY owner
    ORDER BY reclaimable_acres DESC
""").show(truncate=False)

## 8. Persist — Iceberg Table for the Agronomy Stack

One managed table with three geometry columns (field square, pivot
circle, corners) plus the economics. Downstream BI / notebooks / API
services read it directly.

In [ ]:
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{TARGET_DATABASE}")

sedona.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_FQN} AS
    SELECT
        c.field_id, c.pivot_type, c.owner,
        c.field_sq      AS field_geometry,
        c.pivot_circle  AS irrigated_geometry,
        c.corners_geom  AS corners_geometry,
        c.field_acres, c.irrigated_acres, c.corner_acres,
        s.reclaimable_fraction,
        s.annual_uplift_usd,
        s.payback_years,
        s.retrofit_tier
    FROM corner_inventory c
    JOIN scored_corners   s USING (field_id)
""")

print(f"Wrote table: {TARGET_FQN}")
sedona.sql(f"SELECT COUNT(*) AS rows FROM {TARGET_FQN}").show()

## 9. Export GeoJSON — Three Layers for the Map

Field squares, irrigated circles, and corners as separate feature
collections so the dashboard can style each differently (outline /
green fill / amber fill).

In [ ]:
def collect_features(df, geom_col, props):
    rows = df.selectExpr(*props, f"ST_AsGeoJSON({geom_col}) AS geojson").collect()
    feats = []
    for r in rows:
        d = r.asDict()
        gj = d.pop("geojson")
        feats.append({
            "type": "Feature",
            "properties": {k: (float(v) if isinstance(v, (int, float)) else v)
                           for k, v in d.items()},
            "geometry": json.loads(gj),
        })
    return {"type": "FeatureCollection", "features": feats}

base_df = sedona.sql(f"""
    SELECT field_id, pivot_type, owner,
           field_acres, irrigated_acres, corner_acres,
           reclaimable_fraction, annual_uplift_usd, payback_years,
           retrofit_tier,
           field_geometry, irrigated_geometry, corners_geometry
    FROM {TARGET_FQN}
""").cache()

common = ["field_id", "pivot_type", "owner",
          "field_acres", "irrigated_acres", "corner_acres",
          "reclaimable_fraction", "annual_uplift_usd", "payback_years",
          "retrofit_tier"]

for geom_col, fname in [
    ("field_geometry",     "pivot_gap_fields.geojson"),
    ("irrigated_geometry", "pivot_gap_circles.geojson"),
    ("corners_geometry",   "pivot_gap_corners.geojson"),
]:
    fc = collect_features(base_df, geom_col, common)
    path = f"/tmp/{fname}"
    with open(path, "w") as fh:
        json.dump(fc, fh)
    print(f"Wrote {path} ({len(fc['features'])} features)")

## 10. Preview on a Map

In [ ]:
viz = SedonaKepler.create_map()
SedonaKepler.add_df(
    viz,
    base_df.selectExpr("field_id", "owner", "field_acres",
                       "field_geometry AS geometry"),
    name="Field Squares"
)
SedonaKepler.add_df(
    viz,
    base_df.selectExpr("field_id", "pivot_type", "irrigated_acres",
                       "irrigated_geometry AS geometry"),
    name="Irrigated Circles"
)
SedonaKepler.add_df(
    viz,
    base_df.selectExpr("field_id", "corner_acres", "retrofit_tier",
                       "corners_geometry AS geometry"),
    name="Reclaimable Corners"
)
viz

---

## From demo to production

| Stage | Demo | Production |
|---|---|---|
| Field boundary | 12 hand-built squares | `build_and_predict_mosaic_recipe(ModelRecipes.FTW)` + `vectorize_mosaic` on a county / state AOI |
| Irrigated circle | Synthetic `ST_Buffer(center, radius_deg)` | `RS_MapAlgebra` NDVI + growing-season max + threshold + vectorize |
| Pivot-field pairing | Index-matched by construction | Spatial join `ST_Contains(field, ST_Centroid(circle))` + circularity filter |
| Corner geometry | `ST_Difference(field_sq, pivot_circle)` (same) | Same |
| Sink | `org_catalog.pivot_gap_demo.*` | Customer's own managed catalog; scheduled nightly refresh |

## Outputs

| File | Contents |
|---|---|
| `org_catalog.pivot_gap_demo.haskell_corner_inventory` | Iceberg table, one row per field with all three geometries + economics |
| `/tmp/pivot_gap_fields.geojson` | Field squares for the basemap |
| `/tmp/pivot_gap_circles.geojson` | Irrigated circles for the "green" layer |
| `/tmp/pivot_gap_corners.geojson` | Reclaimable corner polygons — the business layer |